# v22 Event Chunk Cache Verification

Run this before training. It checks that event chunk parquet files are readable, samples liquid and illiquid tickers, verifies quote/trade count sanity, validates cached target columns, confirms target horizons are time-based, and checks the training data provider's idle-chunk expansion.

In [ ]:
from pathlib import Path

# Edit these only if your workstation layout changes.
CACHE_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\event_chunks_v1")
CHUNK_MS = 500
MAX_QUOTE_EVENTS = 128
MAX_TRADE_EVENTS = 192
MAX_TOTAL_EVENTS = 256

# Keep this small for a quick smoke check; set to None to scan every parquet file.
CORRUPTION_SCAN_LIMIT = 500

# Number of liquid and illiquid ticker-month files to inspect in detail.
SAMPLE_PER_SIDE = 5

# Horizon chunks to validate. Defaults match v22 target_cache_horizon_chunks.
TARGET_HORIZON_CHUNKS = (20, 40, 60, 120, 240, 600)

PARAM_ROOT = CACHE_ROOT / f"chunk_ms={CHUNK_MS}" / f"mq={MAX_QUOTE_EVENTS}_mt={MAX_TRADE_EVENTS}_m={MAX_TOTAL_EVENTS}"
MANIFEST_PATH = CACHE_ROOT / "preprocess_event_chunks_manifest.jsonl"

print(f"PARAM_ROOT={PARAM_ROOT}")
print(f"MANIFEST_PATH={MANIFEST_PATH}")

In [ ]:
import json
import math
import random
import sys
from datetime import datetime

import numpy as np
import polars as pl

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "research" / "inhouse_transformer" / "v22").exists():
            return path
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.inhouse_transformer.v22.config import DataConfig
from research.inhouse_transformer.v22.data import CHUNK_SUMMARY_COLUMNS, ticker_arrays

assert PARAM_ROOT.exists(), f"Missing chunk parameter folder: {PARAM_ROOT}"
print(f"repo_root={REPO_ROOT}")
print(f"polars={pl.__version__}")

In [ ]:
files = sorted(PARAM_ROOT.glob("ticker=*/*.parquet"))
assert files, f"No parquet files found under {PARAM_ROOT}"

file_rows = []
for path in files:
    stat = path.stat()
    file_rows.append({
        "path": str(path),
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "bytes": stat.st_size,
        "modified": datetime.fromtimestamp(stat.st_mtime),
    })

files_df = pl.DataFrame(file_rows).sort("bytes", descending=True)
print(f"files={len(files):,}")
print(f"total_gb={files_df['bytes'].sum() / 1e9:.2f}")
display(files_df.select("ticker", "year_month", "bytes", "modified").head(10))
display(files_df.select("ticker", "year_month", "bytes", "modified").tail(10))

In [ ]:
if MANIFEST_PATH.exists():
    status_counts = {}
    phase_counts = {}
    failures = []
    manifest_rows = 0
    with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            manifest_rows += 1
            status_counts[row.get("status", "<missing>")] = status_counts.get(row.get("status", "<missing>"), 0) + 1
            phase_counts[row.get("phase", "<missing>")] = phase_counts.get(row.get("phase", "<missing>"), 0) + 1
            if row.get("status") == "failed":
                failures.append(row)
    print(f"manifest_rows={manifest_rows:,}")
    print(f"phase_counts={phase_counts}")
    print(f"status_counts={status_counts}")
    if failures:
        print(f"FAILED rows={len(failures):,}")
        display(pl.DataFrame(failures[:20]))
    else:
        print("No failed rows in manifest.")
else:
    print("Manifest not found. Continuing with parquet-level checks only.")

In [ ]:
def parquet_row_count(path: Path) -> int:
    return int(pl.scan_parquet(str(path)).select(pl.len()).collect().item())

scan_paths = files if CORRUPTION_SCAN_LIMIT is None else files[: min(len(files), CORRUPTION_SCAN_LIMIT)]
bad = []
checked = 0
for path in scan_paths:
    try:
        schema = pl.scan_parquet(str(path)).collect_schema()
        row_count = parquet_row_count(path)
        required = {"ticker", "session_date", "chunk_start_ns", "chunk_end_ns", "quote_values", "trade_values", "event_kinds", "event_indices"}
        missing = sorted(required - set(schema.names()))
        if missing or row_count <= 0:
            bad.append({"path": str(path), "rows": row_count, "missing": missing, "error": "schema_or_empty"})
    except Exception as exc:
        bad.append({"path": str(path), "rows": None, "missing": None, "error": repr(exc)})
    checked += 1

print(f"parquet_files_checked={checked:,}")
if bad:
    display(pl.DataFrame(bad))
    raise RuntimeError(f"Found {len(bad)} unreadable or invalid parquet files.")
print("Parquet readability/schema smoke check passed.")

In [ ]:
top = files_df.head(SAMPLE_PER_SIDE)
bottom = files_df.filter(pl.col("bytes") > 0).tail(SAMPLE_PER_SIDE)
sample_df = pl.concat([top, bottom], how="vertical_relaxed").unique(["path"]).sort("bytes", descending=True)
sample_paths = [Path(value) for value in sample_df["path"].to_list()]
display(sample_df.select("ticker", "year_month", "bytes", "modified"))

In [ ]:
def summarize_chunk_file(path: Path) -> dict:
    df = pl.read_parquet(path)
    target_cols = [name for name in df.columns if name.startswith("target_")]
    null_exprs = [(pl.col(name).null_count() / pl.len()).alias(f"{name}_null_frac") for name in target_cols]
    base = df.select([
        pl.len().alias("rows"),
        pl.col("session_date").n_unique().alias("sessions"),
        pl.col("quote_count").sum().alias("quote_count_sum"),
        pl.col("trade_count").sum().alias("trade_count_sum"),
        pl.col("event_count").sum().alias("event_count_sum"),
        (pl.col("has_quote") > 0).sum().alias("chunks_with_quote"),
        (pl.col("has_trade") > 0).sum().alias("chunks_with_trade"),
        pl.col("latest_mid").min().alias("min_latest_mid"),
        pl.col("latest_mid").max().alias("max_latest_mid"),
    ]).to_dicts()[0]
    nulls = df.select(null_exprs).to_dicts()[0] if null_exprs else {}
    return {
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "bytes": path.stat().st_size,
        "target_columns": len(target_cols),
        **base,
        **{k: round(float(v), 6) for k, v in nulls.items()},
    }

summary = pl.DataFrame([summarize_chunk_file(path) for path in sample_paths])
display(summary)

In [ ]:
def validate_time_based_targets(path: Path, horizons: tuple[int, ...]) -> pl.DataFrame:
    df = pl.read_parquet(path).sort(["session_date", "chunk_start_ns"])
    chunk_ns = CHUNK_MS * 1_000_000
    bounds = df.group_by("session_date").agg(
        pl.col("chunk_start_ns").min().alias("min_chunk_start_ns"),
        pl.col("chunk_start_ns").max().alias("max_chunk_start_ns"),
    )
    quote_grid = (
        bounds.with_columns(
            pl.int_ranges(
                pl.col("min_chunk_start_ns"),
                pl.col("max_chunk_start_ns") + chunk_ns,
                chunk_ns,
            ).alias("chunk_start_ns")
        )
        .explode("chunk_start_ns")
        .select(["session_date", "chunk_start_ns"])
        .join(df.select(["session_date", "chunk_start_ns", "latest_bid", "latest_ask"]), on=["session_date", "chunk_start_ns"], how="left")
        .sort(["session_date", "chunk_start_ns"])
        .with_columns(
            pl.col("latest_bid").forward_fill().over("session_date"),
            pl.col("latest_ask").forward_fill().over("session_date"),
        )
    )
    rows = []
    for horizon in horizons:
        required = [f"target_bid_h{horizon}", f"target_ask_h{horizon}", f"target_mid_h{horizon}"]
        if any(name not in df.columns for name in required):
            rows.append({"horizon_chunks": horizon, "status": "missing_columns"})
            continue
        expected = quote_grid.with_columns(
            pl.col("latest_bid").shift(-horizon).over("session_date").alias("expected_bid"),
            pl.col("latest_ask").shift(-horizon).over("session_date").alias("expected_ask"),
        ).with_columns(((pl.col("expected_bid") + pl.col("expected_ask")) * 0.5).alias("expected_mid"))
        check = df.select(["session_date", "chunk_start_ns", *required]).join(
            expected.select(["session_date", "chunk_start_ns", "expected_bid", "expected_ask", "expected_mid"]),
            on=["session_date", "chunk_start_ns"],
            how="left",
        )
        stats = check.select([
            pl.len().alias("rows"),
            pl.col(required[2]).null_count().alias("target_mid_nulls"),
            (pl.col(required[0]) - pl.col("expected_bid")).abs().max().alias("max_bid_abs_diff"),
            (pl.col(required[1]) - pl.col("expected_ask")).abs().max().alias("max_ask_abs_diff"),
            (pl.col(required[2]) - pl.col("expected_mid")).abs().max().alias("max_mid_abs_diff"),
        ]).to_dicts()[0]
        rows.append({"horizon_chunks": horizon, "horizon_seconds": horizon * CHUNK_MS / 1000.0, "status": "ok", **stats})
    return pl.DataFrame(rows)

target_checks = []
for path in sample_paths:
    check = validate_time_based_targets(path, TARGET_HORIZON_CHUNKS).with_columns(
        pl.lit(path.parent.name.split("=", 1)[1]).alias("ticker"),
        pl.lit(path.stem).alias("year_month"),
    )
    target_checks.append(check)
target_check_df = pl.concat(target_checks, how="vertical_relaxed")
display(target_check_df.select("ticker", "year_month", "horizon_chunks", "horizon_seconds", "status", "rows", "target_mid_nulls", "max_bid_abs_diff", "max_ask_abs_diff", "max_mid_abs_diff"))

bad_targets = target_check_df.filter(
    (pl.col("status") != "ok") |
    (pl.col("max_bid_abs_diff").fill_null(0.0) > 1e-6) |
    (pl.col("max_ask_abs_diff").fill_null(0.0) > 1e-6) |
    (pl.col("max_mid_abs_diff").fill_null(0.0) > 1e-6)
)
if bad_targets.height:
    display(bad_targets)
    raise RuntimeError("Cached target columns do not match dense time-grid shifts.")
print("Time-based target validation passed for sampled files.")

In [ ]:
def inspect_idle_expansion(path: Path) -> dict:
    config = DataConfig(
        cache_root=CACHE_ROOT,
        chunk_ms=CHUNK_MS,
        max_quote_events=MAX_QUOTE_EVENTS,
        max_trade_events=MAX_TRADE_EVENTS,
        max_total_events=MAX_TOTAL_EVENTS,
    )
    frame = pl.read_parquet(path).sort(["session_date", "chunk_start_ns"])
    first_session = frame.get_column("session_date")[0]
    one_session = frame.filter(pl.col("session_date") == first_session)
    arrays = ticker_arrays(one_session, config)
    if arrays is None:
        return {"ticker": path.parent.name.split("=", 1)[1], "year_month": path.stem, "session": first_session, "status": "no_arrays"}
    event_count_idx = CHUNK_SUMMARY_COLUMNS.index("event_count")
    seconds_since_quote_idx = CHUNK_SUMMARY_COLUMNS.index("seconds_since_quote")
    seconds_since_trade_idx = CHUNK_SUMMARY_COLUMNS.index("seconds_since_trade")
    event_counts = arrays["chunk_summary"][:, event_count_idx]
    return {
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "session": first_session,
        "sparse_rows_session": one_session.height,
        "dense_rows_session": int(arrays["chunk_summary"].shape[0]),
        "idle_rows_session": int((event_counts == 0.0).sum()),
        "max_seconds_since_quote": float(arrays["chunk_summary"][:, seconds_since_quote_idx].max()),
        "max_seconds_since_trade": float(arrays["chunk_summary"][:, seconds_since_trade_idx].max()),
        "status": "ok",
    }

idle_rows = [inspect_idle_expansion(path) for path in sample_paths]
idle_df = pl.DataFrame(idle_rows)
display(idle_df)

if idle_df.filter((pl.col("status") == "ok") & (pl.col("dense_rows_session") < pl.col("sparse_rows_session"))).height:
    raise RuntimeError("Dense idle expansion produced fewer rows than sparse input for at least one sample.")
print("Idle expansion check passed for sampled files.")

In [ ]:
print("Verification complete.")
print("Review the displayed tables above. If there were no exceptions, the sampled cache is ready for training.")